In [23]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# %% [markdown]
# ## 1. Summary of Tables and Row Counts

# %%
with duckdb.connect(str(db_path), read_only=True) as conn:
    tables = conn.execute("SELECT table_name FROM duckdb_tables();").fetchall()

    data = []
    for (table_name,) in tables:
        count = conn.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
        data.append({"table_name": table_name, "row_count": count})

    df_summary = (
        pd.DataFrame(data)
        .sort_values(by="row_count", ascending=False)
        .reset_index(drop=True)
    )

display(df_summary)

# %% [markdown]
# ## 2. Detailed EDA: Schema & Preview
# Run the cell below to automatically fetch column names, data types, and preview the first 5 rows for every table in the database.

# %%
with duckdb.connect(str(db_path), read_only=True) as conn:
    for _, row in df_summary.iterrows():
        t_name = row["table_name"]
        t_rows = row["row_count"]

        display(HTML(f"<h3>📦 Table: <code>{t_name}</code> ({t_rows:,} rows)</h3>"))

        # # Get schema info
        # schema_df = conn.execute(f'DESCRIBE "{t_name}"').fetchdf()
        # display(schema_df)

        # Preview top 5 rows
        display(HTML("<strong>Preview (Top 5 rows):</strong>"))
        preview_df = conn.execute(f'SELECT * FROM "{t_name}" LIMIT 5').fetchdf()
        display(preview_df)

        print("-" * 80)

,table_name,row_count
0,bom_observations,10224
1,aemo_wem_dispatch,9792
2,openelectricity_mix,6912
3,aemo_holidays,90


,ts,station_id,region,temp_c,apparent_temp_c,dew_point_c,humidity_pct,wind_speed_kmh,wind_direction_deg,wind_gust_kmh,pressure_hpa,rain_since_9am_mm,cloud_oktas,source,ingested_at,ingest_run_id,_ingest_run_id
0,2026-01-04 06:00:00+06:00,066037,NSW1,26.8,29.3,18.1,59,10.3,55,22.7,1010.2,<NA>,2.6,bom,2026-08-05 20:24:04.001537+06:00,9292fcb6-9d41-40ea-a15a-6ba027448bc5,277ca7cc-ee83-4a7d-b16c-fb4b63e5a135
1,2026-01-04 07:00:00+06:00,066037,NSW1,28.2,32.2,18.9,57,11.1,56,31.0,1009.3,<NA>,6.6,bom,2026-08-05 20:24:04.001537+06:00,9292fcb6-9d41-40ea-a15a-6ba027448bc5,277ca7cc-ee83-4a7d-b16c-fb4b63e5a135
2,2026-01-04 08:00:00+06:00,066037,NSW1,28.1,31.2,19.2,58,14.2,64,36.7,1008.4,<NA>,5.3,bom,2026-08-05 20:24:04.001537+06:00,9292fcb6-9d41-40ea-a15a-6ba027448bc5,277ca7cc-ee83-4a7d-b16c-fb4b63e5a135
3,2026-01-04 09:00:00+06:00,066037,NSW1,28.3,31.4,19.4,59,17.5,71,44.6,1007.4,<NA>,2.4,bom,2026-08-05 20:24:04.001537+06:00,9292fcb6-9d41-40ea-a15a-6ba027448bc5,277ca7cc-ee83-4a7d-b16c-fb4b63e5a135
4,2026-01-04 10:00:00+06:00,066037,NSW1,27.7,31.1,19.1,59,11.4,71,44.3,1007.4,<NA>,7.8,bom,2026-08-05 20:24:04.001537+06:00,9292fcb6-9d41-40ea-a15a-6ba027448bc5,277ca7cc-ee83-4a7d-b16c-fb4b63e5a135


--------------------------------------------------------------------------------


,ts,region,demand_mw,price_mwh,source,ingested_at,ingest_run_id,_ingest_run_id
0,2026-07-01 23:40:00+06:00,WEM,1979.82227,NaN,aemo_wem,2026-08-05 22:33:23.787319+06:00,de184e87-17a1-4907-8d60-9f4e0fd17200,66dd47a4-e56a-4a91-864f-84a45db4f3f4
1,2026-07-01 21:45:00+06:00,WEM,2205.64500,NaN,aemo_wem,2026-08-05 22:33:23.787319+06:00,de184e87-17a1-4907-8d60-9f4e0fd17200,66dd47a4-e56a-4a91-864f-84a45db4f3f4
2,2026-07-01 23:55:00+06:00,WEM,1931.65186,NaN,aemo_wem,2026-08-05 22:33:23.787319+06:00,de184e87-17a1-4907-8d60-9f4e0fd17200,66dd47a4-e56a-4a91-864f-84a45db4f3f4
3,2026-07-01 19:50:00+06:00,WEM,2677.83325,NaN,aemo_wem,2026-08-05 22:33:23.787319+06:00,de184e87-17a1-4907-8d60-9f4e0fd17200,66dd47a4-e56a-4a91-864f-84a45db4f3f4
4,2026-07-01 22:00:00+06:00,WEM,2296.83862,118.73,aemo_wem,2026-08-05 22:33:23.787319+06:00,de184e87-17a1-4907-8d60-9f4e0fd17200,66dd47a4-e56a-4a91-864f-84a45db4f3f4


--------------------------------------------------------------------------------


,ts,battery_discharge_mw,battery_charge_mw,biomass_mw,coal_mw,distillate_mw,gas_mw,hydro_mw,pumped_hydro_mw,solar_rooftop_mw,...,region,total_generation_mw,total_renewable_mw,demand_mw,price_mwh,intensity_kg_per_mwh,source,ingested_at,ingest_run_id,_ingest_run_id
0,2026-07-01 16:00:00+06:00,132.6072,363.8832,23.2122,5061.1551,0.0,43.67,172.9213,0.0,1827.985,...,NSW1,10183.5687,4582.2532,<NA>,<NA>,0.448782,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
1,2026-07-01 16:05:00+06:00,260.6815,240.4171,23.1422,5020.5538,0.0,43.71,204.3357,0.0,1827.985,...,NSW1,10191.8805,4626.5181,<NA>,<NA>,0.445032,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
2,2026-07-01 16:10:00+06:00,-397.6665,399.3063,23.3822,4941.1575,0.0,43.97,170.0763,0.0,1827.985,...,NSW1,9569.7096,4582.9423,<NA>,<NA>,0.466098,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
3,2026-07-01 16:15:00+06:00,-388.2119,399.6667,23.0922,4809.3676,0.0,43.38,149.5140,0.0,1827.985,...,NSW1,9431.6107,4567.4083,<NA>,<NA>,0.460096,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609
4,2026-07-01 16:20:00+06:00,-416.6562,416.6806,23.1922,4738.1018,0.0,43.53,199.7734,0.0,1827.985,...,NSW1,9330.7845,4549.1283,<NA>,<NA>,0.458491,openelectricity,2026-08-05 22:01:46.863771+06:00,c1acc1ba-0cb5-447c-b28d-28c8b6127815,819da310-e741-4a87-9b43-0877bb651609


--------------------------------------------------------------------------------


,date,region,holiday_name,is_workday,source,ingested_at,ingest_run_id,_ingest_run_id
0,2030-01-01,NSW1,New Year's Day,False,aemo_holidays,2026-08-05 20:26:48.005300+06:00,d8ee67ca-28eb-4866-9dfa-a30f7cdead26,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
1,2030-01-26,NSW1,Australia Day,False,aemo_holidays,2026-08-05 20:26:48.005352+06:00,ee896ae2-68fb-4ae7-b660-d21079f803c8,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
2,2030-04-25,NSW1,Anzac Day,False,aemo_holidays,2026-08-05 20:26:48.005369+06:00,347cd951-16c1-4e5d-8b50-e41aa2d39fe0,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
3,2030-12-25,NSW1,Christmas Day,False,aemo_holidays,2026-08-05 20:26:48.005383+06:00,03f90eb4-5307-40f6-b2c1-25613917e482,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94
4,2030-12-26,NSW1,Boxing Day,False,aemo_holidays,2026-08-05 20:26:48.005398+06:00,90bb3fd3-670e-4cf2-8ba3-93162946f443,3b3b8a9a-bc0f-42cf-94cf-b408bf92fc94


--------------------------------------------------------------------------------


In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
from IPython.display import display, HTML

# 1. Locate and load the environment settings
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")
sys.path.insert(0, str(INGESTION_DIR))

from app.core.config import get_settings  # noqa: E402

# 2. Resolve database path
settings = get_settings()
db_path = Path(settings.duckdb_staging_dir)
if not db_path.is_absolute():
    db_path = INGESTION_DIR / db_path
db_path = db_path / "landed.duckdb"

assert db_path.exists(), f"No staging file found at {db_path} -- run an ingest first."

# --- CONFIGURATION ---
TABLE_NAME = "openelectricity_mix"
START_DATE = "2026-04-01"
END_DATE = "2026-08-04"
EXPECTED_DAILY_COUNT = 218
# ---------------------

# %% [markdown]
# ## Completeness Check Query

# %%
query = f"""
SELECT
    cal.record_date,
    COUNT(d.ts) AS total_count
FROM (
    SELECT unnest(generate_series(
        '{START_DATE}'::timestamp,
        '{END_DATE}'::timestamp,
        '1 day'::interval
    ))::date AS record_date
) cal
LEFT JOIN {TABLE_NAME} d
       ON DATE(d.ts) = cal.record_date
      AND d.ts >= '{START_DATE} 00:00:00+00'
      AND d.ts < '{END_DATE} 00:00:00+00'
GROUP BY cal.record_date
HAVING COUNT(d.ts) < {EXPECTED_DAILY_COUNT}
ORDER BY cal.record_date ASC;
"""

with duckdb.connect(str(db_path), read_only=True) as conn:
    df_gaps = conn.execute(query).fetchdf()

display(
    HTML(
        f"<h3>⚠️ Incomplete Days Found (< {EXPECTED_DAILY_COUNT} records): {len(df_gaps)}</h3>"
    )
)
display(df_gaps)

,record_date,total_count
0,2026-05-01,0
1,2026-05-02,0
2,2026-05-03,0
3,2026-05-04,0
4,2026-05-05,0
...,...,...
91,2026-07-31,0
92,2026-08-01,0
93,2026-08-02,0
94,2026-08-03,0
